In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.apm_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.utils.helpers import *
from src.utils.dataScraper import *
from live import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{}

Out Players:
{'BOS': ['Jayson Tatum']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 2 teams with confirmed lineups
Updated 0 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,name
27650,NaN,NaN,NaN,2025-26,1627751,Jakob Poeltl,Jakob,1610612761,TOR,Toronto Raptors,42500136,2026-05-01T00:00:00,TOR vs. CLE,W,21.516667,1,3,0.333,0,0,0.0,0,0,0.00,2,2,4,1,0,2,1,1,3,0,2,0,17.3,0,0,13.0,1,21:31,1,112.0,109.1,109.1,105.0,106.7,106.7,7.1,2.4,2.4,0.063,0.00,25.0,0.100,0.095,0.098,0.0,0.0,0.333,0.333,0.061,0.063,98.78,99.27,82.73,99.27,0.048,44,1.0,3.0,C,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40,87,0.460,13,36,0.361,19,23,0.826,10,28,38,27,16.0,13,9,4,25,21,112,2.0,108.6,110.9,105.9,107.8,2.7,3.0,0.675,1.69,19.0,0.313,0.536,0.433,0.158,0.534,0.577,93.7,91.92,76.60,101,0.533,1610612739,CLE,Cleveland Cavaliers,40,93,0.43,11,41,0.268,19,27,0.704,19,33,52,23,18.0,8,4,9,21,25,110,-2.0,105.9,107.8,108.6,110.9,-2.7,-3.0,0.575,1.28,15.2,0.464,0.688,0.567,0.176,0.489,0.524,93.7,91.92,76.60,102,0.467,1,C,30.0,NaN,NaN,NaN,NaN,1,0.092951,0.046476,0.185902,1,0,Scottie Barnes,Brandon Ingram,RJ Barrett,0,0,2,1,Jakob Poeltl
27651,NaN,NaN,NaN,2025-26,1631288,Jamal Cain,Jamal,1610612753,ORL,Orlando Magic,42500106,2026-05-01T00:00:00,ORL vs. DET,L,20.150000,1,3,0.333,1,2,0.5,0,0,0.00,1,3,4,2,1,1,1,1,4,2,3,-7,15.8,0,0,14.0,1,20:09,1,92.7,92.1,92.1,113.3,105.0,105.0,-20.6,-12.9,-12.9,0.167,2.00,33.3,0.048,0.167,0.103,16.7,16.7,0.500,0.500,0.091,0.096,89.14,92.90,77.42,92.90,0.045,38,1.0,3.0,F,4.14,1.45,2.0,3.0,5.0,15.0,0.0,0.0,10.0,0.0,1.0,0.0,1.0,2.0,0.5,1.0,2.0,0.5,27,78,0.346,9,36,0.250,16,21,0.762,8,30,38,20,11.0,6,4,8,22,21,79,-14.0,87.5,89.8,105.2,105.7,-17.6,-15.9,0.741,1.82,16.5,0.241,0.685,0.463,0.125,0.404,0.453,89.3,88.00,73.33,88,0.403,1610612765,DET,Detroit Pistons,32,80,0.40,9,27,0.333,20,26,0.769,14,38,52,16,11.0,5,8,4,21,22,93,14.0,105.2,105.7,87.5,89.8,17.6,15.9,0.500,1.45,13.4,0.315,0.759,0.537,0.125,0.456,0.509,89.3,88.00,73.33,88,0.597,1,SF,26.0,NaN,NaN,NaN,NaN,1,0.148883,0.099256,0.198511,1,3,Paolo Banchero,Franz Wagner,Desmond Bane,0,0,2,1,Jamal Cain
27652,NaN,NaN,NaN,2025-26,1631222,Jake LaRavia,Jake,1610612747,LAL,Los Angeles 

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260502_145031.json


,home_team,away_team,commence_time,bookmakers
0,Boston Celtics,Philadelphia 76ers,2026-05-02 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Detroit Pistons,Orlando Magic,2026-05-03 19:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Cleveland Cavaliers,Toronto Raptors,2026-05-03 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,San Antonio Spurs,Minnesota Timberwolves,2026-05-05 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,Oklahoma City Thunder,Los Angeles Lakers,2026-05-06 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26


#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-02 14:50:32
US latest pull: 2026-05-02 14:49:27


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Tyrese Maxey,Over,0.5,-137,2026-05-02,2026-05-02T21:50:21Z,2026-05-02 14:50:32
1,PrizePicks,player_points,Joel Embiid,Over,26.5,-137,2026-05-02,2026-05-02T21:50:21Z,2026-05-02 14:50:32
2,PrizePicks,player_points,Joel Embiid,Under,26.5,-137,2026-05-02,2026-05-02T21:50:21Z,2026-05-02 14:50:32
3,PrizePicks,player_points,Tyrese Maxey,Over,24.5,-137,2026-05-02,2026-05-02T21:50:21Z,2026-05-02 14:50:32
4,PrizePicks,player_points,Tyrese Maxey,Under,24.5,-137,2026-05-02,2026-05-02T21:50:21Z,2026-05-02 14:50:32


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-02.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-01-01.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-01-01.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-01-01.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Tyrese Maxey,PTS,31.49,41.26,45.06,0.3627,0.5938,0.9498,11.42,24.50,42.80,"[0.6045808626903848, 0.5237191650853891, 0.823..."
1,Joel Embiid,PTS,22.65,29.02,36.26,0.4028,0.6329,0.9818,9.12,18.37,35.60,"[1.0303058430407943, 1.0252029047415634, 0.744..."
2,Paul George,PTS,23.04,28.55,36.51,0.2790,0.4915,0.7551,6.43,14.03,27.57,"[0.7149404216315307, 0.5468937395058767, 1.283..."
3,VJ Edgecombe,PTS,31.44,39.55,43.30,0.1926,0.4378,0.7398,6.06,17.32,32.03,"[0.4229017566688354, 0.4346614655893007, 0.699..."
4,Quentin Grimes,PTS,18.05,24.14,31.36,0.1854,0.4340,0.7333,3.35,10.48,23.00,"[0.2803738317757009, 0.3481012658227848, 0.519..."
5,Cade Cunningham,PTS,31.24,39.82,44.21,0.3938,0.6288,0.9503,12.30,25.04,42.01,"[0.7396870554765292, 0.8129032258064518, 0.281..."
6,Paolo Banchero,PTS,31.84,39.92,43.90,0.3844,0.5875,0.9088,12.24,23.45,39.89,"[0.3006681514476614, 0.5114401076716016, 0.418..."
7,Desmond Bane,PTS,28.65,36.78,41.42,0.3002,0.5124,0.7897,8.60,18.85,32.71,"[0.6411062225015713, 0.562751228226887, 0.4705..."
8,Tobias Harris,PTS,25.68,35.03,40.27,0.2205,0.4728,0.7585,5.66,16.56,30.55,"[0.620884289746002, 0.4630225080385852, 0.6424..."
9,Jalen Duren,PTS,19.61,26.16,32.87,0.2926,0.5087,0.8194,5.74,13.31,26.94,"[0.6183115338882283, 0.9544008483563096, 0.344..."


In [8]:
from live import adjust_predictions

# Build contexts dict once (using the notebook's get_game_context)
game_contexts = {
    name: get_game_context(base_df, name, team_odds)
    for name in pts_preds["PLAYER_NAME"]
}

# Adjust the model's Q50 predictions with scenario signals
pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
ast_preds

Tyrese Maxey [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 41.3→41.4 (Δ+0.18)  RATE: 0.5938→0.5931 (Δ-0.0006)
Joel Embiid [PTS]  pace_bucket=low_pace  MIN: 29.0→30.9 (Δ+1.85)  RATE: 0.6329→0.6147 (Δ-0.0182)
Paul George [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 28.6→29.5 (Δ+0.99)  RATE: 0.4915→0.4739 (Δ-0.0176)
VJ Edgecombe [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 39.5→39.7 (Δ+0.15)  RATE: 0.4378→0.4660 (Δ+0.0282)
Quentin Grimes [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 24.1→23.9 (Δ-0.28)  RATE: 0.4340→0.4108 (Δ-0.0232)
Cade Cunningham [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 39.8→40.2 (Δ+0.37)  RATE: 0.6288→0.5739 (Δ-0.0549)
Paolo Banchero [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 39.9→40.1 (Δ+0.15)  RATE: 0.5875→0.5663 (Δ-0.0212)
Desmond Bane [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 36.8→36.5 (Δ-0.25)  RATE: 0.5124→0.4676 (Δ-0.0448)
Tobias Harris [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 35.

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY,ADJ_CONTEXT_OK,ADJ_CONTEXT_ERR,ADJ_ACTIVE_STARS,ADJ_STARS_MISSING,ADJ_SPREAD_ROLE,ADJ_CTX_SPREAD,ADJ_CTX_TOTAL,ADJ_MIN_DELTA,ADJ_RATE_DELTA,ADJ_MIN_SHIFT,ADJ_RATE_SHIFT,ADJ_USED_INTERACTION,ADJ_MIN_LOG,ADJ_RATE_LOG
0,Tyrese Maxey,AST,31.67,41.44,45.24,0.0747,0.1474,0.2413,2.37,6.11,10.92,"[0.181625, 0.200534, 0.260176, 0.225155, 0.025...",True,None,3,0,underdog,4.5,203.5,0.1809,-0.00440,0.18,-0.0044,True,"{'stars': (0.6307, 29), 'pace': (-0.1846, 33),...","{'stars': (0.0119, 29), 'pace': (-0.0066, 33),..."
1,Joel Embiid,AST,24.50,30.87,38.11,0.0630,0.1348,0.2369,1.54,4.16,9.03,"[0.222519, 0.10227, 0.092821, 0.181207, 0.0906...",True,None,3,0,underdog,4.5,203.5,1.8504,-0.00025,1.85,-0.0003,False,"{'stars': (1.8054, 29), 'pace': (-0.4272, 18),...","{'stars': (0.0134, 29), 'pace': (0.0011, 18), ..."
2,Quentin Grimes,AST,17.77,23.86,31.08,0.0019,0.0793,0.1560,0.03,1.89,4.85,"[0.09509, 0.116532, 0.033209, 0.030264, 0.0475...",True,None,3,0,underdog,4.5,203.5,-0.2768,-0.01005,-0.28,-0.0101,True,"{'stars': (-0.0368, 26), 'pace': (0.7745, 43),...","{'stars': (-0.0055, 26), 'pace': (0.0024, 43),..."
3,Cade Cunningham,AST,31.61,40.19,44.58,0.1076,0.1909,0.3153,3.40,7.67,14.06,"[0.295395, 0.591545, 0.468915, 0.508138, 0.252...",True,None,3,0,favorite,-8.5,202.5,0.3732,0.01090,0.37,0.0109,True,"{'stars': (-0.1864, 39), 'pace': (0.2492, 45),...","{'stars': (0.0041, 39), 'pace': (-0.0023, 45),..."
4,Anthony Black,AST,16.86,24.40,31.44,0.0135,0.0733,0.1651,0.23,1.79,5.19,"[0.019241, 0.122969, 0.078455, 0.117624, -0.00...",True,None,3,0,underdog,8.5,202.5,-2.5565,-0.00825,-2.56,-0.0083,True,"{'stars': (-1.7541, 29), 'pace': (-2.6009, 46)...","{'stars': (0.0024, 29), 'pace': (-0.0069, 46),..."
5,Scottie Barnes,AST,30.77,38.53,43.99,0.1272,0.2157,0.3483,3.91,8.31,15.32,"[0.55599, 0.460152, 0.310352, 0.284895, 0.2765...",True,None,3,0,underdog,8.5,211.5,-0.0359,0.02720,-0.04,0.0272,True,"{'stars': (-0.4562, 73), 'pace': (0.6982, 46),...","{'stars': (-0.0081, 73), 'pace': (-0.0003, 46)..."
6,James Harden,AST,31.19,38.93,43.61,0.1022,0.1815,0.2891,3.19,7.07,12.61,"[0.194941, 0.171306, 0.481176, 0.368825, 0.114...",True,None,3,0,favorite,-8.5,211.5,-0.5983,0.00660,-0.60,0.0066,True,"{'stars': (0.1584, 61), 'pace': (-0.1522, 51),...","{'stars': (0.0082, 61), 'pace': (-0.008, 51), ..."
7,Jamal Shead,AST,22.49,31.60,38.89,0.0910,0.1754,0.2944,2.05,5.54,11.45,"[0.42576, 0.407823, 0.186564, 0.256235, 0.1926...",True,None,3,0,underdog,8.5,211.5,0.5863,-0.00220,0.59,-0.0022,True,"{'stars': (-0.951, 65), 'pace': (0.8996, 44), ...","{'stars': (-0.0004, 65), 'pace': (-0.0098, 44)..."
8,Evan Mobley,AST,27.10,35.23,41.46,0.0211,0.0786,0.1448,0.57,2.77,6.00,"[0.171337, 0.117419, 0.089361, 0.08793, 0.1726...",True,None,3,0,favorite,-8.5,211.5,-0.4030,-0.01325,-0.40,-0.0132,True,"{'stars': (-0.5956, 74), 'pace': (-0.4919, 46)...","{'stars': (-0.0016, 74), 'pace': (-0.0274, 46)..."
9,Brandon Ingram,AST,19.85,25.19,33.87,0.0166,0.0880,0.1818,0.33,2.22,6.16,"[0.029575, 0.10484, 0.058366, -0.0071, 0.18459...",True,None,3,0,underdog,8.5,211.5,-0.2590,-0.00710,-0.26,-0.0071,True,"{'stars': (-0.2936, 55), 'pace': (-0.2244, 30)...","{'stars': (-0.0021, 55), 'pace': (0.0039, 30),..."


### Get Line Probabilities

In [9]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
127,Shai Gilgeous-Alexander,PTS,31.5,29.39,37.12,41.57,12.36,24.89,42.06,0.468,0.532
62,Luka Garza,REB,2.5,14.27,17.93,23.78,1.76,4.86,10.24,0.757,0.243
60,Donovan Mitchell,REB,4.0,29.53,36.85,42.88,1.83,4.50,10.01,0.818,0.182
23,Paul George,AST,3.5,24.03,29.54,37.50,0.70,2.56,5.57,0.306,0.694
10,Stephon Castle,AST,7.0,26.05,34.99,40.68,4.14,8.47,14.64,0.758,0.242
21,Derrick White,AST,4.5,26.55,35.21,39.98,0.85,3.30,7.27,0.278,0.722
63,Jaylen Brown,REB,7.5,29.42,38.17,42.89,2.54,6.53,11.45,0.281,0.719
19,Paolo Banchero,AST,5.5,31.99,40.07,44.05,1.97,5.07,9.49,0.580,0.420
118,Luguentz Dort,PTS,6.5,15.55,23.30,29.13,1.85,9.16,20.47,0.667,0.333
66,Joel Embiid,PTS,26.5,24.50,30.87,38.11,9.42,18.98,36.72,0.389,0.612


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
underdog_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
underdog_all_lines

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
6,James Harden,AST,6.5,31.19,38.93,43.61,3.19,7.07,12.61,0.567,0.433,AST,Underdog,Toronto Raptors,-8.2,211.2,112.1,5.0,99.22,21.0,-108.0,-110.0,0.519,0.524,6.1,5.0,2.23,-0.4,-1.5,0.179,0.429,0.571,-17.38,9.01,0.4,0.4,0.53,0.73,35.20,4.93,0.27,0.06,7.11,9.0
9,Brandon Ingram,AST,2.5,19.85,25.19,33.87,0.33,2.22,6.16,0.497,0.503,AST,Underdog,Cleveland Cavaliers,8.2,211.2,114.1,15.0,100.70,13.0,-137.0,-137.0,0.578,0.578,3.3,3.0,2.06,0.8,0.5,-0.388,0.651,0.349,12.62,-39.63,0.4,0.6,0.53,0.74,31.49,7.59,0.23,0.06,3.00,9.0
12,Julius Randle,AST,4.5,27.55,35.94,41.01,1.06,4.20,7.49,0.428,0.572,AST,Underdog,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,-105.0,-112.0,0.512,0.528,4.2,4.0,1.55,-0.3,-0.5,0.194,0.423,0.577,-17.41,9.22,0.6,0.4,0.33,0.47,33.35,2.97,0.27,0.04,5.14,7.0
21,Derrick White,AST,4.5,26.55,35.21,39.98,0.85,3.30,7.27,0.278,0.722,AST,Underdog,Philadelphia 76ers,-5.0,204.0,114.4,17.0,100.40,15.0,170.0,-141.0,0.370,0.585,3.3,3.0,1.83,-1.2,-1.5,0.656,0.256,0.744,-30.88,27.17,0.0,0.3,0.40,0.52,33.06,6.46,0.15,0.06,4.43,14.0
25,Joel Embiid,REB,8.5,24.50,30.87,38.11,3.43,7.34,14.61,0.485,0.515,REB,Underdog,Boston Celtics,5.0,204.0,111.7,4.0,95.58,30.0,102.0,-106.0,0.495,0.515,8.5,8.5,3.06,0.0,0.0,0.000,0.500,0.500,1.00,-2.83,0.8,0.5,0.47,0.43,33.62,4.16,0.34,0.04,6.86,7.0
29,Jalen Duren,REB,9.5,20.40,26.95,33.66,3.52,7.79,15.10,0.300,0.700,REB,Underdog,Orlando Magic,-8.5,202.5,113.6,13.0,100.56,14.0,-105.0,-115.0,0.512,0.535,8.5,9.0,0.85,-1.0,-0.5,1.176,0.120,0.880,-76.57,64.52,0.0,0.0,0.27,0.56,29.38,3.32,0.18,0.04,9.08,13.0
34,Duncan Robinson,REB,2.5,21.14,28.18,36.42,0.75,3.21,7.90,0.599,0.401,REB,Underdog,Orlando Magic,-8.5,202.5,113.6,13.0,100.56,14.0,-105.0,-117.0,0.512,0.539,2.3,2.0,2.00,-0.2,-0.5,0.100,0.460,0.540,-10.19,0.15,0.6,0.4,0.40,0.45,27.13,4.38,0.16,0.04,2.62,13.0
36,Jarrett Allen,REB,7.5,21.85,28.79,35.11,3.48,7.95,14.94,0.479,0.521,REB,Underdog,Toronto Raptors,-8.2,211.2,112.1,5.0,99.22,21.0,105.0,-108.0,0.488,0.519,7.3,7.0,4.14,-0.2,-0.5,0.048,0.481,0.519,-1.40,-0.04,0.2,0.4,0.47,0.65,26.24,3.93,0.18,0.07,8.36,11.0
40,Ja'Kobe Walter,REB,3.5,24.81,34.14,42.27,0.73,3.27,9.20,0.456,0.544,REB,Underdog,Cleveland Cavaliers,8.2,211.2,114.1,15.0,100.70,13.0,-125.0,105.0,0.556,0.488,3.2,2.5,2.39,-0.3,-1.0,0.126,0.450,0.550,-19.00,12.75,0.4,0.3,0.47,0.31,29.17,5.70,0.14,0.04,2.82,11.0
47,Jaden McDaniels,REB,4.5,27.55,36.19,42.42,2.20,5.73,11.73,0.739,0.261,REB,Underdog,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,100.0,105.0,0.500,0.488,5.8,6.5,3.29,1.3,2.0,-0.395,0.654,0.346,30.80,-29.07,0.6,0.6,0.53,0.56,34.05,7.08,0.23,0.07,6.00,7.0


In [16]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
prizePicks_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
95,Evan Mobley,PTS,16.5,27.50,35.63,41.86,8.11,17.87,31.45,0.669,0.331,PTS,PrizePicks,Toronto Raptors,-8.0,211.0,112.1,5.0,99.22,21.0,-104.0,-115.0,0.510,0.535,18.2,19.5,6.66,1.7,3.0,-0.255,0.601,0.399,17.89,-25.40,0.6,0.6,0.60,0.61,32.18,5.25,0.21,0.05,18.15,13.0
46,Julius Randle,REB,7.0,27.74,36.13,41.20,2.86,7.03,13.00,0.565,0.435,REB,PrizePicks,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,-137.0,-137.0,0.578,0.578,6.7,7.0,2.31,-0.3,0.0,0.130,0.448,0.552,-22.50,-4.51,0.6,0.4,0.40,0.37,33.35,2.97,0.27,0.04,6.29,7.0
12,Evan Mobley,AST,2.5,27.50,35.63,41.86,0.95,3.27,6.62,0.672,0.328,AST,PrizePicks,Toronto Raptors,-8.0,211.0,112.1,5.0,99.22,21.0,-115.0,110.0,0.535,0.476,3.3,3.0,1.83,0.8,0.5,-0.437,0.669,0.331,25.07,-30.49,0.6,0.7,0.80,0.65,32.18,5.25,0.21,0.05,3.69,13.0
121,Jalen Williams,PTS,16.0,18.67,23.77,30.57,5.45,12.53,25.55,0.522,0.478,PTS,PrizePicks,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-137.0,-137.0,0.578,0.578,17.7,18.0,6.02,1.7,2.0,-0.282,0.611,0.389,5.70,-32.71,0.6,0.7,0.67,0.74,25.25,4.01,0.25,0.05,18.80,5.0
1,Jayson Tatum,AST,5.5,24.53,28.95,38.09,1.90,4.22,9.08,0.481,0.519,AST,PrizePicks,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-132.0,110.0,0.569,0.476,6.9,7.0,2.56,1.4,1.5,-0.547,0.708,0.292,24.44,-38.68,0.6,0.7,0.67,0.53,36.19,4.56,0.27,0.04,6.80,10.0


In [17]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
betr_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
85,Jalen Duren,PTS,12.5,19.61,26.16,32.87,5.74,13.31,26.94,0.458,0.542,PTS,Betr DFS,Orlando Magic,-8.5,202.5,113.6,13.0,100.56,14.0,-125.0,100.0,0.556,0.500,13.4,12.0,5.02,0.9,-0.5,-0.179,0.571,0.429,2.78,-14.20,0.0,0.4,0.53,0.63,29.38,3.32,0.18,0.04,12.69,13.0
95,Evan Mobley,PTS,16.5,27.50,35.63,41.86,8.11,17.87,31.45,0.669,0.331,PTS,Betr DFS,Toronto Raptors,-8.0,211.0,112.1,5.0,99.22,21.0,-104.0,-115.0,0.510,0.535,18.2,19.5,6.66,1.7,3.0,-0.255,0.601,0.399,17.89,-25.40,0.6,0.6,0.60,0.61,32.18,5.25,0.21,0.05,18.15,13.0
32,Tyrese Maxey,REB,3.5,31.49,41.26,45.06,1.35,4.68,9.35,0.533,0.467,REB,Betr DFS,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-125.0,104.0,0.556,0.490,3.9,3.0,2.88,0.4,-0.5,-0.139,0.555,0.445,-0.10,-9.22,0.6,0.4,0.47,0.48,37.96,4.71,0.27,0.05,4.08,13.0
96,Collin Murray-Boyles,PTS,13.5,18.71,24.08,32.14,3.08,11.04,23.45,0.475,0.525,PTS,Betr DFS,Cleveland Cavaliers,8.0,211.0,114.1,15.0,100.70,13.0,-104.0,-110.0,0.510,0.524,13.0,14.5,6.06,-0.5,1.0,0.083,0.467,0.533,-8.40,1.75,0.8,0.6,0.60,0.22,24.85,6.89,0.17,0.07,12.88,8.0
26,Neemias Queta,REB,8.5,15.20,22.30,30.26,2.79,7.47,13.75,0.578,0.422,REB,Betr DFS,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-114.0,105.0,0.533,0.488,8.0,7.5,3.40,0.5,0.0,-0.147,0.558,0.442,4.75,-9.39,0.6,0.5,0.60,0.38,21.60,7.00,0.14,0.03,7.92,13.0


In [18]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
draftKings_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
78,Quentin Grimes,PTS,6.5,18.05,24.14,31.36,3.35,10.48,23.00,0.668,0.332,PTS,DraftKings Pick6,Boston Celtics,7.5,204.5,111.7,4.0,95.58,30.0,-114.0,106.0,0.533,0.485,9.6,6.5,7.31,3.1,0.0,-0.424,0.664,0.336,24.65,-30.78,0.4,0.5,0.60,0.76,22.42,4.14,0.17,0.07,10.31,13.0
23,Payton Pritchard,AST,4.5,23.18,31.00,36.68,1.80,4.71,8.68,0.591,0.409,AST,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,125.0,-133.0,0.444,0.571,5.1,5.0,2.23,0.6,0.5,-0.269,0.606,0.394,36.35,-30.98,0.8,0.7,0.60,0.38,31.62,3.65,0.20,0.05,4.54,13.0
128,Sam Hauser,PTS,6.5,16.79,23.21,31.56,2.00,9.25,22.44,0.708,0.292,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,110.0,-122.0,0.476,0.550,8.6,6.0,5.80,2.1,-0.5,-0.362,0.641,0.359,34.61,-34.67,0.2,0.4,0.60,0.55,23.63,5.00,0.14,0.04,6.15,13.0
72,Jayson Tatum,PTS,23.5,24.53,28.95,38.09,9.38,17.09,35.17,0.311,0.689,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,-103.0,100.0,0.507,0.500,23.3,23.5,3.50,-0.2,0.0,0.057,0.477,0.523,-5.99,4.60,0.6,0.5,0.53,0.59,36.19,4.56,0.27,0.04,25.70,10.0
26,Neemias Queta,REB,8.5,15.20,22.30,30.26,2.79,7.47,13.75,0.578,0.422,REB,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,140.0,-148.0,0.417,0.597,8.0,7.5,3.40,-0.5,-1.0,0.147,0.442,0.558,6.08,-6.50,0.4,0.4,0.47,0.26,21.60,7.00,0.14,0.03,7.92,13.0


In [13]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
23,Payton Pritchard,AST,4.5,23.18,31.00,36.68,1.80,4.71,8.68,0.590,0.410,AST,DraftKings Pick6,Philadelphia 76ers,-7.5,204.5,114.4,17.0,100.40,15.0,125.0,-133.0,0.444,0.571,5.1,5.0,2.23,0.6,0.5,-0.269,0.606,0.394,36.35,-30.98,0.8,0.7,0.60,0.38,31.62,3.65,0.20,0.05,4.54,13.0
64,Max Strus,REB,4.5,18.28,24.95,32.00,1.71,4.57,9.70,0.597,0.403,REB,Betr DFS,Toronto Raptors,-8.0,211.0,112.1,5.0,99.22,21.0,113.0,-123.0,0.469,0.552,4.6,4.5,1.71,0.1,0.0,-0.058,0.523,0.477,11.40,-13.52,0.6,0.5,0.53,0.52,23.75,3.99,0.16,0.05,4.38,8.0
16,Julius Randle,AST,4.5,27.74,36.13,41.20,1.52,4.80,8.19,0.565,0.435,AST,Underdog,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,-105.0,-112.0,0.512,0.528,4.2,4.0,1.55,-0.3,-0.5,0.194,0.423,0.577,-17.41,9.22,0.6,0.4,0.33,0.47,33.35,2.97,0.27,0.04,5.14,7.0
100,Max Strus,PTS,8.5,18.28,24.95,32.00,2.57,10.83,23.52,0.555,0.445,PTS,Betr DFS,Toronto Raptors,-8.0,211.0,112.1,5.0,99.22,21.0,-110.0,-103.0,0.524,0.507,8.4,7.0,6.80,-0.1,-1.5,0.015,0.494,0.506,-5.69,-0.27,0.2,0.3,0.47,0.57,23.75,3.99,0.16,0.05,10.75,8.0
93,Scottie Barnes,PTS,21.5,30.81,38.57,44.03,10.04,21.00,36.47,0.496,0.504,PTS,Betr DFS,Cleveland Cavaliers,8.0,211.0,114.1,15.0,100.70,13.0,103.0,-115.0,0.493,0.535,21.6,22.0,6.02,0.1,0.5,-0.017,0.507,0.493,2.92,-7.83,0.8,0.5,0.40,0.33,35.67,6.59,0.24,0.06,20.92,13.0


### Get top EVs for 2 legs

In [14]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 97  |  Pairs: 37  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [15]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 44  |  Pairs: 15  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 23  |  Pairs: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 61  |  Pairs: 27  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [18]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 97  |  Triples: 536  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 44  |  Triples: 116  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 61  |  Triples: 153  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 23  |  Triples: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
